# Experiment 1 — the benchmark

**Steps 4, 5 and 6 of 8 — portfolio, backtest, attribution — for one idea.**

**It produces** a book in `Portfolio/`, a track record in `Backtest/` and a breakdown of the
return in `Attribution/`. **It prevents** a good signal in a portfolio nobody could hold, paper
returns that real trading would have erased, and factor beta sold as alpha.

Experiment 1 is the **declared benchmark**: the yardstick every later experiment is measured
against. It is not a null. It is a real strategy with a real return, so beating it is a higher bar
than beating a no-model control — and **its rules freeze once `FINDINGS_1.md` reports**, because a
change to them invalidates every comparison in `RESULTS.md`. Improvements go into a new experiment.

The hypothesis is in [`BLUEPRINT_1.md`](BLUEPRINT_1.md), **written before this notebook's rule**.
The running log is [`JOURNAL_1.md`](JOURNAL_1.md); the results that survive are in
[`FINDINGS_1.md`](FINDINGS_1.md).

**This notebook is empty by design.** Each section below says what is expected in it. Write the
cells.

## Position in the pipeline

This notebook is **only the strategy**. The universe and the data are built by earlier stages and
are simply read here:

```
Universe/universe.ipynb   ->  Security_Master.csv
Data/curator.py           ->  Data/Curator/Time_Series/     m_* + c_*
Data/refinery.py          ->  Data/Refinery/Time_Series/    + r_*      <- this notebook reads here
Data/analyzer.ipynb       ->  the measurements the blueprint's predictions came from
        |
        v
experiment_1.ipynb        ->  Portfolio/  ->  Backtest/  ->  Attribution/
```

**What this notebook does not do.** It does not download anything, profile the universe, or compute
a signal. A number about the data itself belongs in the Universe or Data stage — that separation is
what keeps every experiment comparable, because all of them read the identical panel.

**Four modules beside this notebook are shared by every experiment** — `securities_panel.py`,
`portfolio_construction.py`, `backtest_engine.py` and `attribution_analysis.py`, one per Lab
library — and no strategy column is named in any of them. The benchmark loads the panel through
`securities_panel.py` like every later experiment, so the comparison is on the rule and nothing
else.

## The section contract

Every experiment notebook has the same shape, so anyone who has read one can read all of them.
**Everything below section 2 is strategy-agnostic**, given the three objects that section produces.

| Section | Contains |
| --- | --- |
| 0 · Setup | paths, and **the strategy's columns** — the only strategy names in this notebook outside the rule |
| 1 · The panel | load the refined files and reshape them |
| 2 · The rule | selection, sizing, timing. **The one cell you write** |
| 2.1 · Invariants | what every rule must pass, whatever it is |
| 3 · Construction | the book, and the diagnostics a person would run it on |
| 4 · Backtest | one engine pass, guarded import, reports-and-skips without a licence |
| 5 · Attribution | where the return came from, guarded the same way |
| 6 · Counterfactuals | the arms that price who earned the idiosyncratic share |
| 7 · Verdict | what it concluded, **in words** |
| Handoff | what the next stage consumes, and what this one left open |
| 8 · Verify | assertions that raise when the output is wrong: the invariants, the weight file read back, the days each engine run valued against its window's trading days |

## 0 · Setup

Paths, and the strategy's columns. **Declare them here, not in a shared module**, so a signal never
becomes every later experiment's default without anyone deciding it. Read only the columns the
strategy consumes.

Three price columns do **three different jobs**, and getting them out of step is silent — the
backtest P&L and the attribution would quietly run on different bases:

| Role | Basis | Why |
| --- | --- | --- |
| Daily mark | dividend-and-split adjusted close | total-return valuation between rebalances |
| Fill | dividend-and-split adjusted VWAP | the price a trade actually gets |
| Commission | **unadjusted** VWAP | per-share cents ride on the unadjusted share count |

A provider may return its VWAP columns as null, which is why the Curator reconstructs both VWAPs as
`c_*`, and nothing below reads a provider's own VWAP column, whichever provider `Data/curator.py`
asks for.

In [ ]:
# EXAMPLE-ONLY CELL
import os
import pathlib
import sys

import pandas

NOTEBOOK_DIRECTORY = pathlib.Path.cwd()
REPOSITORY_ROOT = next(
    parent
    for parent in (NOTEBOOK_DIRECTORY, *NOTEBOOK_DIRECTORY.parents)
    if (parent / "Universe").is_dir()
)
os.chdir(REPOSITORY_ROOT)
sys.path.insert(0, str(REPOSITORY_ROOT / "Experiments"))

import attribution_analysis  # noqa: E402 - the path above has to exist first
import backtest_engine  # noqa: E402
import portfolio_construction  # noqa: E402
import securities_panel  # noqa: E402

EXPERIMENT_DIRECTORY = REPOSITORY_ROOT / "Experiments" / "Experiment_1"

# The strategy's columns, named here and nowhere else. Three prices do three different jobs.
SIGNAL_COLUMN = "r_trend_50_200"
RANKING_COLUMN = "r_liquidity_rank"
MARK_COLUMN = "m_close_dividend_and_split_adjusted"
FILL_COLUMN = "c_vwap_dividend_and_split_adjusted"

# The rule's settings, from BLUEPRINT_1.md, which was committed before this cell was written.
BOOK_SIZE = 30
REBALANCE_BAND = 0.10
# Excluded by name: no price file at all, or an adjusted price that multiplies by more than six in
# a day, which is a bad print rather than a return. Both are blocking rows in Data_Issues.csv.
EXCLUDED_IDENTIFIERS = (
    "CIT",
    "FMC",
    "LCI",
    "MIC",
    "PARA",
)
POINT_IN_TIME_START = pandas.Timestamp("2017-01-03")
# The long run wants 2002, and the securities allow it: 572 of them have prices back to 2001. What
# does not is the cash proxy, an exchange-traded fund that launched on 2002-07-30, and a book whose
# cash has no price is a book the engine cannot value. The window is clipped to the instruments
# rather than to the ambition.
LONG_START = max(
    pandas.Timestamp("2002-01-02"),
    backtest_engine.earliest_priceable_date(),
)
WINDOW_END = pandas.Timestamp("2026-06-01")

print(f"repository root: {REPOSITORY_ROOT}")
print(f"engine installed: {backtest_engine.ENGINE_INSTALLED}")
print(f"attribution installed: {attribution_analysis.LIBRARY_INSTALLED}")

## 1 · The panel

Load the refined files, resolve **one position per security**, and reshape to matrices.

A point-in-time universe contains renamed securities: two identifiers sharing one identity, each
carrying part of the history. Left alone they are two independent positions and the book
double-counts at the changeover. Key positions by a stable identity — an ISIN where the seed
carries one — falling back to the identifier itself, and where two legs overlap on a date let the
leg still reporting later win.

The long panel then becomes one wide `dates x securities` matrix per input, which is what makes the
whole rule in section 2 a handful of vectorised lines instead of a loop over files.

In [ ]:
# EXAMPLE-ONLY CELL
matrices = securities_panel.load_matrices((
    SIGNAL_COLUMN,
    RANKING_COLUMN,
    MARK_COLUMN,
    FILL_COLUMN,
))
signal = matrices[SIGNAL_COLUMN]
ranking = matrices[RANKING_COLUMN]
mark = matrices[MARK_COLUMN]
fill = matrices[FILL_COLUMN]
returns = mark.pct_change(fill_method=None)

# Membership, point in time: the index's own daily holdings, mapped onto positions rather than
# listings, because a company that changed ticker is one position here.
position_keys = securities_panel.read_position_keys()
holdings = pandas.read_csv(
    attribution_analysis.BENCHMARK_HOLDINGS_PATH,
    parse_dates=["date_column"],
    dayfirst=True,
).set_index("date_column")
holdings.columns = [position_keys.get(column, column) for column in holdings.columns]
in_index_by_listing = holdings > 0
in_index = in_index_by_listing.T.groupby(level=0).any().T
membership = in_index.reindex(
    index=mark.index,
    columns=mark.columns,
).ffill()
membership = membership.where(membership.notna(), False).astype(bool)

print(f"panel: {mark.shape[0]} dates x {mark.shape[1]} positions")
print(f"index membership known from {holdings.index.min().date()} to {holdings.index.max().date()}")
print(f"positions in the index on the last known date: {int(in_index.iloc[-1].sum())}")

## 2 · The rule — the one cell you write

Three statements, in order: **who is eligible**, **how much of each**, and **when to trade**.

**The contract this cell must satisfy** — everything below reads exactly these three objects:

| Object | Type | Meaning |
| --- | --- | --- |
| `selected_matrix` | `dates x securities` boolean | what the book holds on each day |
| `REBALANCE_DATES` | a date index | the days the book is re-struck |
| `target_weights` | `REBALANCE_DATES x securities` float, rows summing to **at most** 1.0 | the book on each of those days |

**Rows sum to at most one, not to exactly one.** A book that must be fully invested cannot express
a defensive strategy. The residual becomes cash in section 3.1, parked in a real priced instrument,
because the engine's weight file has no cash row of its own.

**Sizing is a seam, not a decision buried in the rule.** Hand the eligible set and a returns
history that has already been cut off before today to a weighting function in
`portfolio_construction.py`, and swapping equal weight for inverse volatility, hierarchical risk
parity or any other method of the KaxaNuk Portfolio Construction library is one line — the module
builds the library's method on that cut history, one rebalance date at a time. That is what makes
two experiments comparable rather than merely adjacent.

**Trade only when something changed.** A signal that has not moved is not a reason to pay
commission.

### Two look-aheads, both stated plainly

**The lag.** The eligible set used on rebalance date *t* is the one observed at *t-1*, and the fill
happens at *t*'s price — a full day between the signal and the fill.

**The delisting exit.** A security that delists must be sold on the **last day it still has a fill
price**, and knowing that day is its last requires seeing the next one. This is the standard
backtest compromise — the alternative, carrying a position that can never be exited, is a larger
distortion — and it is implemented by making a name ineligible on that final day, so the set
changes, the rebalance fires, and the position is sold while a price still exists.

In [ ]:
# EXAMPLE-ONLY CELL
# The rule, in three statements: who is eligible, how much of each, and when to trade.
# Written after BLUEPRINT_1.md was committed, which the history shows.
excluded = [identifier for identifier in EXCLUDED_IDENTIFIERS if identifier in mark.columns]
tradable = mark.notna() & fill.notna()


def build_book(start, end, require_membership, use_filter=True):
    """
    The same rule over a window, with or without the index's own membership.

    `use_filter=False` is the control: the same thirty most traded names, the same sizing and the
    same band, with the trend condition switched off. Claim 1 is about the filter, so this is the
    comparison that tests it -- beating the index only says the book worked.
    """
    in_window = (mark.index >= start) & (mark.index <= end)
    # 1 - who is eligible: in an uptrend, tradable today, not excluded, and in the index when we
    #     know who was in it.
    eligible = ((signal > 0) if use_filter else tradable) & tradable
    kept = eligible.drop(columns=excluded, errors="ignore")
    eligible = kept.reindex(columns=mark.columns).fillna(False).astype(bool)

    if require_membership:
        eligible = eligible & membership

    # ... of those, the thirty most traded. The rank is descending, so 1 is the most traded.
    standing = ranking.where(eligible)
    held = standing.rank(axis=1, ascending=False, method="first") <= BOOK_SIZE
    selected = (held & eligible).loc[in_window]
    # The set used on a rebalance date is the one observed the day before.
    lagged = portfolio_construction.lag_eligibility(selected)
    # 2 - when to trade: only when reaching the new target would move a tenth of the book.
    rebalance_dates = portfolio_construction.select_rebalance_dates(lagged, REBALANCE_BAND)
    # 3 - how much of each: one thirtieth, with the cap doing the work when fewer than thirty
    #     qualify, so the rest stays in cash rather than concentrating the book.
    weights = portfolio_construction.build_weights(
        lagged,
        returns.loc[mark.index <= end],
        rebalance_dates,
        "equal_weight",
        1.0 / BOOK_SIZE,
        1,
    )

    return lagged, rebalance_dates, weights


selected_matrix, REBALANCE_DATES, target_weights = build_book(
    POINT_IN_TIME_START,
    WINDOW_END,
    require_membership=True,
)
print(f"point-in-time book: {len(REBALANCE_DATES)} rebalances, "
      f"{selected_matrix.index.min().date()} to {selected_matrix.index.max().date()}")
per_rebalance = target_weights.gt(0).sum(axis=1)
print(f"holdings per rebalance: min {per_rebalance.min()}, mean {per_rebalance.mean():.1f}")

## 2.1 · Invariants

Cheap to check here, expensive to discover inside a P&L. **Every rule must pass these unchanged**,
whatever the strategy is:

- no book is more than fully invested, and none is negatively invested;
- no negative weights, if the strategy is long-only;
- **every security paid for today had its signal on at the prior close** — check the signal itself,
  not the composed eligibility, because the signal is the thing that had to exist in advance;
- every security bought is tradable on the day it is bought, so a fill price exists;
- nothing is still held on a day after it stopped being tradable.

In [ ]:
# EXAMPLE-ONLY CELL
# Cheap here, expensive inside a P&L. Every rule must pass these unchanged.
invested = target_weights.sum(axis=1)
checks = {
    "no book more than fully invested": bool((invested <= 1.0 + 1e-9).all()),
    "no book negatively invested": bool((invested >= -1e-9).all()),
    "no negative weights": bool((target_weights >= -1e-9).all().all()),
    "every position had its signal on at the prior close": None,
    "every position tradable on the day it is bought": None,
    "nothing held after it stopped being tradable": None,
}

# The signal itself, not the composed eligibility: the signal is the thing that had to exist in
# advance, and checking the composition would only prove the composition is self-consistent.
signal_yesterday = (signal > 0).shift(1).fillna(False)
bought = target_weights > 0
checks["every position had its signal on at the prior close"] = bool(
    (~bought | signal_yesterday.reindex_like(bought)).all().all()
)
checks["every position tradable on the day it is bought"] = bool(
    (~bought | tradable.reindex_like(bought)).all().all()
)
still_held = selected_matrix & ~tradable.reindex_like(selected_matrix)
checks["nothing held after it stopped being tradable"] = bool((~still_held).all().all())

for name, passed in checks.items():
    print(f"{'PASS' if passed else 'FAIL'}  {name}")

## 3 · Construction — is this a book you would actually run?

**This is where step 4, Portfolio Construction, lives.** Four properties, each with a failure mode
a performance chart would hide:

| Property | What a bad value would mean |
| --- | --- |
| Invested share over time | the eligibility column is not doing what the analyzer says it does |
| Trigger frequency and turnover | the rule fires so often that this is a transaction-cost question, not an alpha one |
| Holdings and concentration | a "diversified" label on a book that is one or two positions |
| Group drift | the strategy is a disguised bet on one group rather than a rotation between them |

Measure turnover **target-to-target**. The realised figure is lower, because between rebalances the
winners drift up on their own; that calculation needs drifted weights and belongs to the backtest.

> **The blueprint's predictions about the *shape* of the book, rather than about its return, are
> settled here — before any backtest.** They are the first ones that can be wrong, and the cheapest
> to be wrong about.

## 3.1 · Write the deliverables

Two views of the same book, because two readers need it: a **long, human-readable** one with names
and classifications attached, and a **wide, identifier-keyed** `portfolio_weights.csv` — the
backtest engine's input, which looks each identifier up in the market-data folder and so has to
speak in identifiers, not in stitched positions.

**This is where cash becomes a position.** Everything above lets a book be less than fully
invested; here the residual becomes a weight in the cash proxy, so the engine charges commission on
going to cash and earns the yield while there. A strategy whose defining move is *sell everything*
has to pay for it.

In [ ]:
# EXAMPLE-ONLY CELL
# Is this a book you would actually run? Four properties, each hiding a different failure.
daily_book = target_weights.reindex(selected_matrix.index).ffill()
invested_share = daily_book.sum(axis=1)
holdings_count = daily_book.gt(0).sum(axis=1)
# Turnover target to target: the realised figure is lower, because winners drift up on their own,
# and that calculation needs the drifted weights the engine returns.
turnover = target_weights.diff().abs().sum(axis=1) / 2
years = (selected_matrix.index.max() - selected_matrix.index.min()).days / 365.25
# The effective number of positions: one over the sum of squared weights. It equals the count when
# the book is equally weighted and less when it is concentrated.
effective_positions = 1 / daily_book.pow(2).sum(axis=1).replace(0, pandas.NA)

construction = pandas.Series({
    "rebalances": len(REBALANCE_DATES),
    "rebalances per year": round(len(REBALANCE_DATES) / years, 1),
    "mean one-way turnover per rebalance": round(turnover.mean(), 3),
    "annual turnover": round(turnover.sum() / years, 2),
    "mean invested share": round(invested_share.mean(), 3),
    "lowest invested share": round(invested_share.min(), 3),
    "days below 90% invested": int((invested_share < 0.9).sum()),
    "mean holdings": round(holdings_count.mean(), 1),
    "mean effective positions": round(effective_positions.mean(), 1),
})
print(construction.to_string())

# Group drift: is this a rotation, or one sector wearing a rotation's clothes?
master = pandas.read_csv("Universe/Security_Master.csv").set_index("main_identifier")
sector_of = master["sector"].reindex(daily_book.columns).fillna("unknown")
by_sector = daily_book.T.groupby(sector_of).sum().T
yearly_sector = by_sector.groupby(by_sector.index.year).mean()
print()
print("average weight by sector, by year (top 6 sectors):")
top_sectors = yearly_sector.mean().sort_values(ascending=False).head(6).index
print(yearly_sector[top_sectors].round(3).to_string())

In [ ]:
# EXAMPLE-ONLY CELL
# Two views of the same book. The long one is for a person; the wide one is the engine's input and
# has to speak in identifiers, because that is what its market-data folder is named by.
readable = target_weights.stack()
readable = readable[readable > 0].rename("weight").reset_index()
readable.columns = ["rebalance_date", "position", "weight"]
readable["name"] = readable["position"].map(master["name"])
readable["sector"] = readable["position"].map(master["sector"])
readable.to_csv(EXPERIMENT_DIRECTORY / "Portfolio" / "holdings_readable.csv", index=False)

by_identifier = securities_panel.expand_to_identifiers(target_weights)
weight_file = backtest_engine.write_weight_file(by_identifier, EXPERIMENT_DIRECTORY)
written = pandas.read_csv(weight_file, index_col=0)
print(f"{weight_file.name}: {written.shape[0]} identifiers x {written.shape[1]} rebalance dates")
print(f"every column sums to 1: {bool((written.sum(axis=0).sub(1.0).abs() < 1e-6).all())}")
cash_row = written.loc[backtest_engine.CASH_IDENTIFIER]
print(f"cash weight: first {cash_row.iloc[0]:.3f}, mean {cash_row.mean():.3f}")

## 4 · Backtest — KaxaNuk Backtest Engine

**In plain words:** run the rules over history, with costs, without peeking ahead.

The weight file goes to the licensed engine, which simulates the book share by share: it fills at a
real price, charges per-share commission on the unadjusted price, holds integer share counts and a
cash reserve, marks the portfolio daily between rebalances, and compares against the benchmarks.

**This is the only backtest in the repository**, and results are accepted **net** or not at all.
Clip the window to the shortest benchmark up front rather than discovering it as a crash, and report
which benchmark bound it.

> **Guard the import.** The engine installs from KaxaNuk's licensed index rather than PyPI, so this
> section reports what is missing and skips without it. Everything in `Portfolio/` is already
> written and does not depend on the engine — a clone with no licence gets a real book and no
> numbers, by design.

In [ ]:
# EXAMPLE-ONLY CELL
# Costs, stated in BLUEPRINT_1.md rather than defaulted. The cash reserve is not a strategy choice:
# weights summing to exactly one leave nothing to pay commission with, and the engine refuses the
# first rebalance rather than quietly overdrawing. A real book holds the same buffer.
INITIAL_CAPITAL = 1_000_000
COMMISSION_CENTS = 0.1
SLIPPAGE_BASIS_POINTS = 5.0
# Two percent, not half a percent: the long window truncated at 2003 with 0.5 and at 2009 with 1.0,
# because a book that churns between stocks and cash during a crash pays commission faster than a
# thin reserve refills. Every variant uses the same figure, or the comparison is about the reserve.
CASH_RESERVE = 0.02

# The last variant changes nothing but the commission. BLUEPRINT_1.md froze 0.1, describing it as
# a tenth of a cent; the engine charges about eight cents a share for it, roughly twenty times a
# real retail rate. The blueprint is not edited -- it is the hypothesis -- so the frozen figure
# stays the headline and the realistic one is reported beside it.
REALISTIC_COMMISSION_CENTS = 0.005
VARIANTS = (
    (
        "filter on, point in time",
        POINT_IN_TIME_START,
        True,
        True,
        "portfolio_weights",
        COMMISSION_CENTS,
    ),
    (
        "filter off, point in time",
        POINT_IN_TIME_START,
        True,
        False,
        "portfolio_weights_filter_off",
        COMMISSION_CENTS,
    ),
    (
        "filter on, long window",
        LONG_START,
        False,
        True,
        "portfolio_weights_long",
        COMMISSION_CENTS,
    ),
    (
        "filter on, realistic costs",
        POINT_IN_TIME_START,
        True,
        True,
        "portfolio_weights_cheap",
        REALISTIC_COMMISSION_CENTS,
    ),
)
runs = {}

if not backtest_engine.ENGINE_INSTALLED:
    print("step 5 skipped: the KaxaNuk Backtest Engine is not installed")
else:
    for label, start, require_membership, use_filter, name, commission in VARIANTS:
        held, dates, weights = build_book(start, WINDOW_END, require_membership, use_filter)
        by_identifier = securities_panel.expand_to_identifiers(weights)
        backtest_engine.write_weight_file(by_identifier, EXPERIMENT_DIRECTORY, name)
        configuration = backtest_engine.build_configuration(
            EXPERIMENT_DIRECTORY,
            start.date(),
            WINDOW_END.date(),
            INITIAL_CAPITAL,
            commission,
            SLIPPAGE_BASIS_POINTS,
            CASH_RESERVE,
            name,
        )
        result = backtest_engine.run_backtest(EXPERIMENT_DIRECTORY, configuration)
        runs[label] = {
            "result": result,
            "rebalances": len(dates),
            "window": backtest_engine.describe_window(result, WINDOW_END.date()),
        }
        print(f"{label}: success={result.success}, {runs[label]['window']}")

In [ ]:
# EXAMPLE-ONLY CELL
# Every figure below comes from the engine. There is no second simulator in this repository.
if len(runs) == 0:
    print("step 5 skipped: no backtest to summarise")
else:
    rows = {}

    for label, run in runs.items():
        statistics = run["result"].data["portfolio_stats"]
        rows[label] = {
            "CAGR": statistics["Annualized Return (CAGR)"],
            "volatility": statistics["Annualized Volatility"],
            "Sharpe": statistics["Portfolio Sharpe Ratio"],
            "max drawdown": statistics["Max Drawdown"],
            "alpha vs index": statistics.get("Alpha"),
            "information ratio": statistics.get("Information Ratio"),
            "commissions": statistics["Total Commissions"],
            "slippage": statistics["Total Slippage Costs"],
            "rebalances": run["rebalances"],
        }

    benchmark_statistics = runs["filter on, point in time"]["result"].data["benchmark_stats"]
    rows["the index, same window"] = {
        "CAGR": benchmark_statistics["Annualized Return (CAGR)"],
        "volatility": benchmark_statistics["Annualized Volatility"],
        "Sharpe": benchmark_statistics["Portfolio Sharpe Ratio"],
        "max drawdown": benchmark_statistics["Max Drawdown"],
    }
    summary = pandas.DataFrame(rows).T
    print(summary.round(4).to_string())

    # The one comparison claim 1 is about: the filter against no filter, on the same names.
    with_filter = rows["filter on, point in time"]["CAGR"]
    without_filter = rows["filter off, point in time"]["CAGR"]
    print()
    difference = (with_filter - without_filter) * 100
    print(f"filter on minus filter off: {difference:.2f} points a year")

## 5 · Attribution — KaxaNuk Attribution Analysis

**In plain words:** which part of the return did you actually earn?

The backtest says *how much* the book made; attribution says **where it came from**:

- **Brinson-Fachler**, the first cut: active return into an **allocation** effect — being
  overweight the right groups — and a **selection** effect, picking the right securities inside
  them. The exact lever that moved.
- **A factor model**, the second layer: excess return into **compensated factor tilts** — beta,
  momentum, residual volatility, liquidity — and **idiosyncratic** alpha, what was earned on
  purpose rather than by accident.
- **Brinson-Fachler again, on the residual**, the third pass: the selection story sharpens, and
  it says whether the Sharpe survives once the factor turns.

**What it settles and what it does not** is in [`../../AGENTS.md`](../../AGENTS.md) — including the
four counterfactual books that answer what the factor model cannot.

**Two inputs are supplied by hand**, from `Data/Curator/Benchmarks/` and `Data/Curator/Factors/` —
the benchmark's weights and returns, and the factor returns. No price provider sells them. Getting
their layout wrong makes the loader read the attribution transposed rather than fail, so shape them
in one place and say what is missing before trying.

**The book arrives daily.** The attribution library rejects a weight file that is not a daily
series, so it reads the book as the engine held it each trading day, drift included, from
`Backtest/` — never `portfolio_weights.csv`, which holds only the rebalance dates.

**What binds the window.** The attribution period is the intersection of the factor files and the
benchmark holdings, so it is usually *shorter* than the backtest. The two sets of numbers describe
different periods and must not be compared directly. Record both windows in `FINDINGS_1.md`.

In [ ]:
# EXAMPLE-ONLY CELL
missing = attribution_analysis.report_missing_inputs()

if len(runs) == 0:
    missing.append("no backtest to attribute: step 5 was skipped")

ATTRIBUTION_READY = len(missing) == 0

if not ATTRIBUTION_READY:
    print("step 6 skipped:", "; ".join(missing))
else:
    import kaxanuk.attribution_analysis.performance_attribution

    headline = runs["filter on, point in time"]["result"]
    daily_weights = backtest_engine.read_daily_weights(headline)
    benchmark_weights = attribution_analysis.load_benchmark_weights(daily_weights.index)
    # The benchmark is compared whole: every constituent the book does not hold enters at zero
    # weight, each with its own price series. A book naming only what it holds is compared against
    # the fraction of the index it happens to own, and the difference comes back as alpha.
    book = attribution_analysis.widen_to_benchmark(
        daily_weights,
        benchmark_weights,
        backtest_engine.BENCHMARK_IDENTIFIER,
    )
    benchmark = benchmark_weights.reindex(columns=book.columns).fillna(0.0)
    # A blocking row in the register applies to every stage, not only to the rule. The book already
    # excludes these by name; the benchmark does not, and one of them -- PARA, whose adjusted price
    # multiplies by 1,773 on 2021-02-12 -- contributed 160 percentage points to the index's
    # reconstructed return in a single day, which would have been reported as the book's shortfall.
    book = book.drop(columns=list(EXCLUDED_IDENTIFIERS), errors="ignore")
    benchmark = benchmark.drop(columns=list(EXCLUDED_IDENTIFIERS), errors="ignore")
    benchmark = benchmark.div(benchmark.sum(axis=1), axis=0).fillna(0.0)
    universe = tuple(book.columns)
    asset_returns = attribution_analysis.load_asset_returns(universe, book.index)
    factor_returns = attribution_analysis.load_factor_returns()

    # The attribution window is the intersection of what the supplied files cover, which is shorter
    # than the backtest. The two sets of numbers describe different periods.
    factor_dates = factor_returns["f_market"].index
    window = book.index[(book.index >= factor_dates.min()) & (book.index <= factor_dates.max())]
    print(f"backtest window:    {book.index.min().date()} to {book.index.max().date()}")
    print(f"attribution window: {window.min().date()} to {window.max().date()}")
    print(f"securities priced: {asset_returns.shape[1]} of {len(universe)} the two books name")

In [ ]:
# EXAMPLE-ONLY CELL
# First cut: Brinson-Fachler. Active return into allocation, selection and interaction.
if ATTRIBUTION_READY:
    brinson = kaxanuk.attribution_analysis.performance_attribution.BrinstonFachlerArrowAttribution(
        attribution_analysis.to_arrow(asset_returns.loc[window]),
        attribution_analysis.to_arrow(book.loc[window]),
        attribution_analysis.to_arrow(benchmark.loc[window]),
        date_column=attribution_analysis.DATE_HEADER,
    )
    # The methods compute in place and return nothing: the tables live on the object.
    brinson.time_series_calculation()
    brinson_daily = brinson.df.to_pandas().set_index("date")
    brinson_totals = brinson_daily[["alpha", "allocation", "selection", "interaction"]].sum() * 100
    print("Brinson-Fachler, summed over the window, in percentage points:")
    print(brinson_totals.round(2).to_string())
    print()
    print("This library's first cut is per asset, not per group: the book and the index hold")
    print("the same securities, so a return difference inside a group has nowhere to land and")
    print("the active return falls into allocation and interaction. The group story is the")
    print("third pass, on the residual.")

In [ ]:
# EXAMPLE-ONLY CELL
# Second layer: the factor model. Excess return into compensated tilts and idiosyncratic alpha.
if ATTRIBUTION_READY:
    by_factor = {}

    for name, frame in factor_returns.items():
        by_factor[name] = attribution_analysis.to_arrow(frame.loc[window.min():window.max()])
    factor_model = kaxanuk.attribution_analysis.performance_attribution.KNFMArrowAttribution(
        attribution_analysis.to_arrow(book.loc[window]),
        by_factor,
        attribution_analysis.to_arrow(asset_returns.loc[window]),
        date_column=attribution_analysis.DATE_HEADER,
    )
    factor_model.multifactor_attribution()
    factor_daily = factor_model.portfolio_attribution_ts.to_pandas()
    factor_totals = factor_daily.select_dtypes("number").sum() * 100
    totals_are = attribution_analysis.RESERVED_FACTOR_NAMES
    reserved = [name for name in factor_totals.index if name in totals_are]
    priced = [name for name in factor_totals.index if name not in totals_are]
    print("Factor model, summed over the window, in percentage points:")
    print(factor_totals[priced].round(2).sort_values(ascending=False).to_string())
    print()
    print("The reserved series, which are totals rather than factors:")
    print(factor_totals[reserved].round(2).to_string())

## 6 · Counterfactuals — who earned the idiosyncratic share

The factor model leaves part of the book's excess return unexplained. That is a number, not an
answer: the book makes choices the index does not. **Each counterfactual removes exactly one of
those and keeps the rest**, which is the only way the question stops being an inference. The
engine can already price all of them; `AGENTS.md`, under *What attribution must report*, names four
follow-ups.

<!-- example: begin -->

Here the factor model leaves 45.5 points of the book's 159.5 points of excess return unexplained,
and the book makes four choices the index does not — it holds thirty names rather than six
hundred, it picks them by traded value, it requires an uptrend, and it weights them equally. Three
of the four follow-ups are run here.

<!-- example: end -->

In [ ]:
# EXAMPLE-ONLY CELL
# Three counterfactual books, and the reason for each, written before any of them was priced.
#
#   the equalised control   filter off, rebalanced on the FILTERED book's own dates. The plain
#                           control fires 11 times against 87, so it is not one lever different:
#                           it also trades a seventh as often and drifts far further into whatever
#                           has been winning. Forcing the dates leaves the trend condition as the
#                           only difference between the two books.
#   the plain control       already priced in section 4; only its attribution is new. Read against
#                           the equalised one it prices the drift, not the filter.
#   the random books        thirty names drawn from the index's own members on the same dates,
#                           equal weight, same band. It removes the liquidity ranking and keeps
#                           everything else. One draw is one sample, so five seeds, and the spread
#                           is the result -- a single random book that happens to beat is noise.
#
# What each outcome would mean, stated now rather than after the fact:
#   - the equalised control near 45.5   the filter is not the source, and the ranking is the
#                                       remaining candidate.
#   - the equalised control well below  the filter earns the idiosyncratic share after all, and
#                                       the control's higher return is factor exposure.
#   - the random books near zero        the ranking is the source.
#   - the random books near 45.5        neither is: the number is what this decomposition assigns
#                                       to any equally weighted thirty-name book, and no selection
#                                       claim survives it.
#
# None of these is a candidate for the headline. They are diagnostics, and they are in the trial
# count because a reader cannot discount what they cannot see.
RANDOM_SEEDS = (11, 22, 33, 44, 55)
in_point_in_time = (mark.index >= POINT_IN_TIME_START) & (mark.index <= WINDOW_END)
window_returns = returns.loc[in_point_in_time]

# The equalised control: the filter-off eligibility, the filtered book's dates.
held_without_filter, own_dates, _ = build_book(
    POINT_IN_TIME_START,
    WINDOW_END,
    require_membership=True,
    use_filter=False,
)
equalised_weights = portfolio_construction.build_weights(
    held_without_filter,
    window_returns,
    REBALANCE_DATES,
    "equal_weight",
    1.0 / BOOK_SIZE,
    1,
)
print(f"equalised control: {len(REBALANCE_DATES)} rebalances, against the plain control's "
      f"{len(own_dates)}")

# The random books: the same dates, the same sizing, no ranking at all.
pool = (tradable & membership).drop(columns=excluded, errors="ignore")
pool = pool.reindex(columns=mark.columns).fillna(False).astype(bool)
lagged_pool = portfolio_construction.lag_eligibility(pool.loc[in_point_in_time])
random_weights = {}

for seed in RANDOM_SEEDS:
    rows = {}

    for position, date in enumerate(REBALANCE_DATES):
        available = lagged_pool.loc[date]
        names = available[available].index.to_series()
        # The rule holds cash rather than concentrating when fewer than thirty qualify, and the
        # lagged pool is empty on day one by construction. The random books do the same thing, or
        # they would be a different book on exactly the dates the rule is most defensive.
        draw_size = min(BOOK_SIZE, len(names))
        row = pandas.Series(0.0, index=lagged_pool.columns)

        if draw_size > 0:
            drawn = names.sample(n=draw_size, random_state=seed * 1000 + position)
            row[drawn.index] = 1.0 / BOOK_SIZE

        rows[date] = row

    random_weights[seed] = pandas.DataFrame(rows).transpose()

depth = lagged_pool.loc[REBALANCE_DATES].sum(axis=1)
print(f"the pool the random books draw from: {depth.min()} to {depth.max()} names per date")

In [ ]:
# EXAMPLE-ONLY CELL
# Priced by the same engine, on the same window, at the same costs. Nothing here may differ from
# the headline run except the book itself.
if backtest_engine.ENGINE_INSTALLED:
    counterfactuals = {"equalised control": equalised_weights}

    for seed in RANDOM_SEEDS:
        counterfactuals[f"random {seed}"] = random_weights[seed]

    for label, weights in counterfactuals.items():
        name = "portfolio_weights_" + label.replace(" ", "_")
        by_identifier = securities_panel.expand_to_identifiers(weights)
        backtest_engine.write_weight_file(by_identifier, EXPERIMENT_DIRECTORY, name)
        configuration = backtest_engine.build_configuration(
            EXPERIMENT_DIRECTORY,
            POINT_IN_TIME_START.date(),
            WINDOW_END.date(),
            INITIAL_CAPITAL,
            COMMISSION_CENTS,
            SLIPPAGE_BASIS_POINTS,
            CASH_RESERVE,
            name,
        )
        result = backtest_engine.run_backtest(EXPERIMENT_DIRECTORY, configuration)
        runs[label] = {
            "result": result,
            "rebalances": len(REBALANCE_DATES),
            "window": backtest_engine.describe_window(result, WINDOW_END.date()),
        }
        statistics = result.data["portfolio_stats"]
        print(f"{label}: CAGR {statistics['Annualized Return (CAGR)']:.4f}, "
              f"Sharpe {statistics['Portfolio Sharpe Ratio']:.3f}, {runs[label]['window']}")

In [ ]:
# EXAMPLE-ONLY CELL
# The factor model on every arm, treated exactly as the headline book was: widened to the whole
# benchmark, blocking names dropped, the same factor files over the same window. Only the factor
# model is run -- this library's first cut is per asset, so its selection number cannot answer a
# question about which choice earned the return.


def decompose(priced):
    """
    One book's excess return split into compensated factor exposure and what is left.

    The treatment is the headline book's, so the arms are comparable with it and with each other;
    a book widened differently would produce a different alpha for the same holdings.
    """
    weights = backtest_engine.read_daily_weights(priced)
    widened = attribution_analysis.widen_to_benchmark(
        weights,
        benchmark_weights,
        backtest_engine.BENCHMARK_IDENTIFIER,
    )
    cleaned = widened.drop(columns=list(EXCLUDED_IDENTIFIERS), errors="ignore")
    aligned = cleaned.reindex(columns=asset_returns.columns).fillna(0.0)
    unpriced = cleaned.drop(columns=asset_returns.columns, errors="ignore")
    model = kaxanuk.attribution_analysis.performance_attribution.KNFMArrowAttribution(
        attribution_analysis.to_arrow(aligned.loc[window]),
        by_factor,
        attribution_analysis.to_arrow(asset_returns.loc[window]),
        date_column=attribution_analysis.DATE_HEADER,
    )
    model.multifactor_attribution()
    daily = model.portfolio_attribution_ts.to_pandas()
    totals = daily.select_dtypes("number").sum() * 100
    # A held name with no price series would leave the book silently under water, so the weight
    # that fell out is reported beside the decomposition rather than trusted to be zero.
    totals["weight unpriced"] = float(unpriced.abs().to_numpy().sum())

    return totals


if ATTRIBUTION_READY:
    arms = {"the rule": headline, "the plain control": runs["filter off, point in time"]["result"]}

    for label in counterfactuals:
        arms[label] = runs[label]["result"]

    decomposition = pandas.DataFrame({label: decompose(priced) for label, priced in arms.items()})
    interesting = [
        "f_total_excess_returns",
        "f_total_factor_returns",
        "f_idyo_returns",
        "f_market",
        "momentum",
        "weight unpriced",
    ]
    present = [name for name in interesting if name in decomposition.index]
    print("Factor model by arm, percentage points over the attribution window:")
    print(decomposition.loc[present].round(2).to_string())
    print()
    random_share = decomposition.loc["f_idyo_returns", [f"random {seed}" for seed in RANDOM_SEEDS]]
    print(f"idiosyncratic points, the rule: {decomposition.loc['f_idyo_returns', 'the rule']:.1f}")
    print(f"the equalised control: {decomposition.loc['f_idyo_returns', 'equalised control']:.1f}")
    print(f"random books: {random_share.min():.1f} to {random_share.max():.1f}, "
          f"mean {random_share.mean():.1f}")

## 7 · Verdict

**In words.** A notebook that ends in a number and no sentence gets read as whatever the reader
hoped.

Three sentences: does the book work; what attribution says about why; what the next experiment
should change. Then copy the numbers into [`FINDINGS_1.md`](FINDINGS_1.md) — **that file is the
record, this notebook is the method.**

If sections 4 and 5 reported "not installed", this notebook has produced a book and no result,
which is the honest outcome and not a failure.

## Handoff

| Output | Consumed by |
| --- | --- |
| `Portfolio/portfolio_weights.csv` | the backtest engine |
| `Portfolio/` — the readable book and the summaries | humans, and `FINDINGS_1.md` |
| `Backtest/` — the track record, and the book's daily weights | the attribution library, `FINDINGS_1.md`, and the comparison baseline for every later experiment |
| `Attribution/` | `FINDINGS_1.md`, and the comparison baseline for every later experiment |

Later experiments read the **same** panel, over the same window, with the same costs and the same
rebalancing convention, and change only the selection or the weighting — which is what makes the
comparison against this benchmark meaningful.

## Open items to carry forward

| # | Item | Why it matters |
| --- | --- | --- |
| 1 | **No result without the licensed engines.** | Without them the book is built but its performance is not measured, and the blueprint's return predictions stay open. |
| 2 | **Turnover is target-to-target, not realised.** | The realised figure is lower. The engine's own series is the one to quote. |
| 3 | **The attribution window is shorter than the backtest**, bound by the supplied files' coverage. | The two sets of numbers describe different periods. |
| 4 | **Delisting exits use one day of hindsight.** | Inert on a universe of live securities; load-bearing on any universe that retains delisted names. |
| 5 | **Cash is a real instrument**, so going flat costs commission and earns a yield. | A strategy that trades to cash often is partly a bet on the front end of the curve. Ask attribution about it. |
| 6 | **Every lever the benchmark declines** — a weight cap, a minimum holding count, risk-aware sizing — is a later experiment, and each has to beat this book to earn its place. | Complexity is added one lever at a time. |

In [ ]:
# EXAMPLE-ONLY CELL
# Three sentences, because a notebook that ends in a number and no sentence gets read as whatever
# the reader hoped. The record is FINDINGS_1.md; this notebook is the method.
print(
    "Does it work? Against the index yes -- 17.85% against 14.71%, Sharpe 0.861 against 0.774,\n"
    "drawdown 30.5% against 33.8%. Against the control that differs in exactly one thing, no:\n"
    "the equalised control earns 18.97%, so the filter costs 1.12 points a year, and a 9.3-point\n"
    "shallower drawdown is what it buys with them.\n"
)
print(
    "Why? Seventy-one percent of the excess return is factor exposure and more than half of the\n"
    "whole is market beta, at a beta of 1.028. The momentum loading came free with the winners\n"
    "themselves. Of the 45.5 idiosyncratic points, the counterfactuals put about 12.5 in a\n"
    "baseline any equally weighted thirty-name book earns here, about 28 in the liquidity\n"
    "ranking, and about 5 in the trend filter.\n"
)
print(
    "What next? Draw far more random books: five seeds prove the baseline is not zero and span\n"
    "43 points, and every share above rests on it. Then the rebalancing band as a curve, which\n"
    "claim 4 has never tested -- a new experiment, because this rule froze when FINDINGS_1.md\n"
    "reported."
)

## 8 · Verify

**Assertions that raise when this experiment's output is wrong.** Section 2.1 prints its
invariants; this section raises on them, and on what the notebook wrote, so a run that reaches the
last cell is one whose book and numbers hold. At the least:

- every invariant of section 2.1 holds;
- `Portfolio/portfolio_weights.csv`, read back, has one column per rebalance date, each summing to
  one with the cash proxy, and no negative weight;
- **every engine run valued the window it was asked for**: the days it valued, counted against the
  trading days in that window — never against a fixed floor, which a run that stopped years early
  can clear. Nearly every trading day valued, and none missing from the window's end, where a
  truncated run loses them. Skipped, as section 4 is, when the engine is not installed;
- the attribution window lies inside the backtest's, and every arm has an idiosyncratic figure.
  Skipped, as section 5 is, when attribution did not run.

In [ ]:
# EXAMPLE-ONLY CELL
# The book: the invariants section 2.1 printed, raised here, and the weight file read back from disk
# rather than from the frame that wrote it.
written_weights = pandas.read_csv(weight_file, index_col=0)
column_totals = written_weights.sum(axis=0)
book_verifications = {
    **checks,
    "one column per rebalance date": written_weights.shape[1] == len(REBALANCE_DATES),
    "every column sums to one, cash included": bool(
        ((column_totals - 1.0).abs() <= 0.0001).all()
    ),
    "the cash proxy has a row": backtest_engine.CASH_IDENTIFIER in written_weights.index,
    "no negative weight": bool((written_weights >= 0.0).all(axis=None)),
}
failed_book = [
    name
    for name, passed in book_verifications.items()
    if not passed
]

if len(failed_book) > 0:
    message = f"the book failed verification: {'; '.join(failed_book)}"

    raise AssertionError(message)

print(f"verified: {len(book_verifications)} checks on the book and {weight_file.name}")

In [ ]:
# EXAMPLE-ONLY CELL
# Every engine run, counted against the trading days of the window it was asked for. A run that
# stops valuing the book still reports success and summarises the stub, and a fixed floor of days
# is cleared by a long run that stopped years early; the window's own days are not. A day or two
# at the edges is the engine's calendar, not a truncation, which loses everything after it.
VALUED_SHARE_REQUIRED = 0.99
UNVALUED_DAYS_AT_END_ALLOWED = 5
run_starts = {
    label: start
    for label, start, *_ in VARIANTS
}
run_verifications = {}

for label, run in runs.items():
    run_start = run_starts.get(label, POINT_IN_TIME_START)
    run_days = mark.index[(mark.index >= run_start) & (mark.index <= WINDOW_END)]
    valued_days = pandas.DatetimeIndex(backtest_engine.read_daily_weights(run["result"]).index)
    valued_in_window = run_days.intersection(valued_days)
    valued_share = len(valued_in_window) / len(run_days)
    unvalued_at_end = run_days[run_days > valued_days.max()]
    run_verifications[f"{label}: {valued_share:.1%} of the window's trading days valued"] = (
        valued_share >= VALUED_SHARE_REQUIRED
    )
    run_verifications[f"{label}: {len(unvalued_at_end)} trading days unvalued at the end"] = (
        len(unvalued_at_end) <= UNVALUED_DAYS_AT_END_ALLOWED
    )

if len(runs) == 0:
    print("engine checks skipped, as section 4 was: the KaxaNuk Backtest Engine is not installed")

if ATTRIBUTION_READY:
    run_verifications["the attribution window lies inside the backtest's"] = bool(
        len(window) > 0
        and window.min() >= book.index.min()
        and window.max() <= book.index.max()
    )
    run_verifications["every arm has an idiosyncratic figure"] = bool(
        decomposition.loc["f_idyo_returns"].notna().all()
    )

failed_runs = [
    name
    for name, passed in run_verifications.items()
    if not passed
]

if len(failed_runs) > 0:
    message = f"the runs failed verification: {'; '.join(failed_runs)}"

    raise AssertionError(message)

checked_runs = len(runs)
print(f"verified: {len(run_verifications)} checks on {checked_runs} engine runs")